# Identify sequences from new virus DBs to mine

### VIRE

In [ ]:
%%bash
# download metadata
# http://spire-dev.embl.de/vire/api/download_metadata/?environment=host_associated&data_type=metadata&format=txt

# download sequences 
# wget https://swifter.embl.de/~fullam/vire/archives/host_associated_genomes.tar -O vire_host_associated_genomes.tar
# tar -xOf vire_host_associated_genomes.tar > vire_host_associated_genomes.fna.gz

In [ ]:
import polars as pl

# load metadata and filter to those from human samples
vire_metadata = pl.read_csv('vire_metadata_env_host_associated.tsv', separator='\t', ignore_errors=True)
vire_metadata_human = vire_metadata.filter(pl.col('microntology').str.contains('human'))

# print number of human-associated viruses in VIRE
print("Number of human-associated viruses in VIRE: ", vire_metadata_human.height)
print("Number of human-associated HQ+ viruses in VIRE: ", vire_metadata_human.filter(pl.col('checkv_completeness') >= 90).height)
print("Number of human-associated Complete viruses in VIRE: ", vire_metadata_human.filter(pl.col('checkv_completeness_method').str.contains('TR')).height)
# 1,002,084 total human-associated viruses in VIRE

# write ids to file
vire_metadata_human.select('genome_id').write_csv('vire_human_virus_ids.txt', include_header=False)

Number of human-associated viruses in VIRE:  1002084
Number of human-associated HQ+ viruses in VIRE:  477101
Number of human-associated Complete viruses in VIRE:  226852


In [ ]:
%%bash
# extract human viruses from full vire dataset
seqkit grep \
    vire_host_associated_genomes.fna.gz \
    --pattern-file vire_human_virus_ids.txt \
    --threads 16 \
    --out-file vire_human_virus_genomes.fna.gz

### OAVGC

In [ ]:
%%bash
# download OAVGC genomes
wget https://zenodo.org/records/18896747/files/virus.fna.gz?download=1 -O oavgc_virus.fna.gz

# download oavgc metadata
wget https://zenodo.org/records/18896747/files/virus.ckv.gz?download=1 -O oavgc.ckv.gz

In [ ]:
import polars as pl

# load oavgc checkv data
oavgc_checkv = pl.read_csv('oavgc.ckv.gz', separator='\t', ignore_errors=True, has_header=False)
oavgc_checkv

# # print number of HQ/Complete viruses in oavgc
print("Number of HQ/Complete viruses in oavgc: ", oavgc_checkv.filter(pl.col('column_10') >= 90).height)
print("Number of Complete viruses in oavgc: ", oavgc_checkv.filter(pl.col('column_11').str.contains('TR')).height)

### All OAVGC sequences are human and will be mined

Number of HQ/Complete viruses in oavgc:  141460
Number of Complete viruses in oavgc:  58265


### MAGIC

In [ ]:
%%bash
# downloaded sequence metadata
# https://www.cell.com/cms/10.1016/j.chom.2024.10.017/attachment/a24c56fd-7372-49b3-bc86-0a0f0b00333e/mmc11.xlsx

# downloaded sample metadata
# https://www.cell.com/cms/10.1016/j.chom.2024.10.017/attachment/4bbc740d-a72a-4796-a769-2309e41543ad/mmc3.xlsx

# download magic sequences
# wget 'https://zenodo.org/records/13989345/files/MAGIC_vMAGs.tar.gz?download=1' -O magic_vMAGs.tar.gz
# tar -xzf magic_vMAGs.tar.gz \
#     --wildcards 'MAGIC_vMAGs/*/*/*/*.fa' \
#     -O > magic_vMAGs.fna
# pigz magic_vMAGs.fna

In [ ]:
# count number of HQ+ viruse in MAGIC
magic_sequence_metadata = pl.read_excel('magic_sequence_metadata.xlsx', read_options={"header_row": 38})
print("Number of MQ+ viruses in MAGIC: ", magic_sequence_metadata.filter(pl.col('completeness') >= 50).height)
print("Number of HQ+ viruses in MAGIC: ", magic_sequence_metadata.filter(pl.col('completeness') >= 90).height)
print("Number of Complete viruses in MAGIC: ", magic_sequence_metadata.filter(pl.col('completeness_method').str.contains('TR')).height)

### All MAGIC sequences are human and will be mined

Could not determine dtype for column 9, falling back to string


Number of MQ+ viruses in MAGIC:  191641
Number of HQ+ viruses in MAGIC:  97694
Number of Complete viruses in MAGIC:  7445


### metaVR

In [ ]:
%%bash
# # download metadata 
# wget https://portal.nersc.gov/cfs/m342/METAVR/METAVR_main_table.parquet

# # download sequences
# wget https://portal.nersc.gov/cfs/m342/METAVR/METAVR.fna.zst

In [13]:
import polars as pl

metavr_metadata = (
    pl.scan_parquet('METAVR_main_table.parquet')
    # filter to human genomes
    .filter(
        (pl.col('Ecosystem').str.contains('uman')) |
        (pl.col('Ecosystem_Category').str.contains('uman')) |
        (pl.col('Ecosystem_Type').str.contains('uman')) |
        (pl.col('Ecosystem_Subtype').str.contains('uman')) |
        (pl.col('Specific_Ecosystem').str.contains('uman'))
    )
    # remove imgvr4 and UHGV sequences
    .filter(~pl.col('discovery').is_in(['geNomad v1.1.0 (2022)', 'UHGV']))
    .collect()
)

# print a preview of the virus counts
print("Number of human-associated viruses in metaVR: ", metavr_metadata.height)
print("Number of human-associated HQ+ viruses in metaVR: ", metavr_metadata.filter(pl.col('completeness') >= 90).height)
print("Number of human-associated Complete viruses in metaVR: ", metavr_metadata.filter(pl.col('completeness_method').str.contains('TR')).height)

# write out human virus IDs to a file
metavr_metadata[['uvig']].write_csv('metavr_human_virus_ids.txt', include_header=False)

Number of human-associated viruses in metaVR:  68515
Number of human-associated HQ+ viruses in metaVR:  29127
Number of human-associated Complete viruses in metaVR:  6639


In [ ]:
%%bash
# extract human viruses from full vire dataset
seqkit grep \
    METAVR.fna.zst \
    --pattern-file metavr_human_virus_ids.txt \
    --threads 4 \
    --id-regexp "^(.*?)\|" \
    --out-file metavr_human_virus_genomes.fna.gz

### NCBI Virus

In [ ]:
%%bash
# download ncbi virus sequence metadata
wget https://ftp.ncbi.nlm.nih.gov/genomes/Viruses/AllNuclMetadata/AllNuclMetadata.csv.gz

In [ ]:
import polars as pl

# load ncbi virus metadata
ncbi_virus_metadata = (
    pl.scan_csv('AllNuclMetadata.csv.gz', ignore_errors=True)
        .select(['#Accession', 'Nuc_Completeness', 'Host', 'Species', 'Isolation_Source', 'SRA_Accession', 'BioSample'])
        .collect()
)

# print number of viruses in NCBI
print("Number of viruses in NCBI: ",
    ncbi_virus_metadata
        .height
)
ncbi_human_non_covid_viruses = (
    ncbi_virus_metadata
        .filter(pl.col('Species') != 'Betacoronavirus pandemicum')
        .filter(pl.col('Nuc_Completeness') == 'complete')
        .filter(pl.col('Host') == 'Homo sapiens')
)
print("Number of complete, non-covid, human viruses in NCBI: ",
    ncbi_human_non_covid_viruses
        .height
)

# write out human non-covid virus IDs to a file
ncbi_human_non_covid_viruses[['#Accession']].write_csv('ncbi_human_non_covid_viruses.txt', include_header=False)

Number of viruses in NCBI:  14953226
Number of complete, non-covid, human viruses in NCBI:  86214


In [ ]:
%%bash
# download human non-covid virus sequences
datasets download virus genome accession --inputfile ncbi_human_non_covid_viruses.txt --filename ncbi_human_non_covid_viruses.zip

# convert to fasta
unzip -j ncbi_human_non_covid_viruses.zip ncbi_dataset/data/genomic.fna -d . && mv genomic.fna ncbi_human_non_covid_viruses.fna

In [ ]:
%%bash
# download sars-cov-2 human complete viruses
datasets download virus genome taxon SARS-CoV-2 --complete-only --include none --host "homo sapiens" --filename sars_cov_2_human_complete.zip

# convert to tsv
dataformat tsv virus-genome --package sars_cov_2_human_complete.zip > sars_cov_2_human_complete.tsv

In [8]:
import polars as pl

# load sars-cov-2 metadata
sars_cov_2_metadata = (
    pl.scan_csv('sars_cov_2_human_complete.tsv', separator='\t', ignore_errors=True)
        .select(['Accession', 'Virus Pangolin Classification', 'Completeness'])
        # .unique('Virus Pangolin Classification')
        .collect()
)

# print number of viruses in sars-cov-2
print("Number of viruses in sars-cov-2: ", sars_cov_2_metadata.height)

# identify one virus per pangolin lineage
sars_cov2_one_per_pango = (
    sars_cov_2_metadata
        .filter(pl.col('Completeness') == 'COMPLETE')
        .group_by('Virus Pangolin Classification')
        .first()
)
print("Number of pango lineages in sars-cov-2: ", sars_cov2_one_per_pango.height)

# write out sars-cov-2 virus IDs to a file
sars_cov2_one_per_pango[['Accession']].write_csv('sars_cov2_one_per_pango.txt', include_header=False)

Number of viruses in sars-cov-2:  3153685
Number of pango lineages in sars-cov-2:  4875


In [ ]:
%%bash
# download human covid sequences (one per pangolin lineage)
datasets download virus genome accession --inputfile sars_cov2_one_per_pango.txt --filename sars_cov2_one_per_pango.zip

# convert to fasta
unzip -j sars_cov2_one_per_pango.zip ncbi_dataset/data/genomic.fna -d . && mv genomic.fna ncbi_virus_sars_cov2_one_per_pango.fna

### Update CheckV with new RefSeq viruses

In [ ]:
%%bash
# download RefSeq viruses
datasets download virus genome taxon "Viruses" --refseq --complete-only --filename refseq_complete_human_viruses.zip

# convert to fasta
unzip -j refseq_complete_human_viruses.zip ncbi_dataset/data/genomic.fna -d . && mv genomic.fna ncbi_virus_refseq_complete_human_viruses.fna

# build ref DB
echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s18/uhvdb_cf_results/uhvdb_2026-04-03/checkv_db/checkv_db_2026-04-03/genome_db/checkv_reps.fna" > ref_kdb.txt
kmer-db \
    build \
    -k 25 \
    -f 0.2 \
    -t 16 \
    -multisample-fasta \
    ref_kdb.txt \
    ref.kdb

# compare query to ref
echo "ncbi_virus_refseq_complete_human_viruses.fna" > query_kdb.txt

kmer-db \
    new2all \
    -sparse \
    -min num-kmers:20 \
    -min ani-shorter:0.95 \
    -t 16 \
    -multisample-fasta \
    ref.kdb \
    query_kdb.txt \
    query_v_ref.csv

# convert output
kmer-db \
    distance \
    ani-shorter \
    -sparse \
    -min 0.95 \
    -t 16 \
    query_v_ref.csv \
    query_v_ref.dist.csv

/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/bin/kmerdb_new2all_to_lzani.py \
    -i query_v_ref.dist.csv \
    -o query_v_ref.dist_mod.csv

cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s18/uhvdb_cf_results/uhvdb_2026-04-03/checkv_db/checkv_db_2026-04-03//genome_db/checkv_reps.fna > ref_query.combined.fna
cat ncbi_virus_refseq_complete_human_viruses.fna >> ref_query.combined.fna

# align with LZ-ANI
lz-ani \
    all2all \
    --in-fasta ref_query.combined.fna \
    -o ncbi_virus_refseq_complete_human_viruses.lzani.tsv \
    --out-format query,reference,ani,qcov,rcov \
    -t 16 \
    --multisample-fasta true \
    --out-type tsv \
    --flt-kmerdb query_v_ref.dist_mod.csv 0.95

# extract new species
csvtk filter2 \
    ncbi_virus_refseq_complete_human_viruses.lzani.tsv  \
    --tabs \
    --filter '( $ani >= 0.95 ) && ( $qcov >= 0.85 || $rcov >= 0.85 )' | \
csvtk cut \
    --tabs \
    --fields query | \
csvtk uniq \
    --tabs \
    --out-file ncbi_virus_refseq_complete_human_viruses.checkv_matches.tsv

seqkit grep \
    ncbi_virus_refseq_complete_human_viruses.fna \
    --threads 16 \
    --invert-match \
    --pattern-file ncbi_virus_refseq_complete_human_viruses.checkv_matches.tsv \
    --out-file ncbi_virus_refseq_complete_human_viruses.novel_checkv.fna.gz
# 5,255 new viruses

# update CheckV database with new viruses
gzip -c -d ncbi_virus_refseq_complete_human_viruses.novel_checkv.fna.gz > new_checkv_genomes.fna

checkv \
    update_database \
    /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s18/uhvdb_cf_results/uhvdb_2026-04-03/checkv_db/checkv_db_2026-04-03 \
    checkv_db_2026-04-03_w_refseq \
    new_checkv_genomes.fna \
    --threads 16

### Run UHVDB/toolkit

In [ ]:
%%bash
nextflow run /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit2 \
    -profile uw_hyak \
    -w /gscratch/scrubbed/carsonjm/uhvdb_v6 \
    -resume \
    --input /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript-update/figure_1/uhvdb_v6_samplesheet.csv \
    --outdir /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript-update/figure_1/results \
    --dbdir /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit2/databases \
    --run_update \
    --hyak_partition='stf' \
    --hyak_queue='cpu-g2'

In [1]:
import polars as pl

# number of sequences mined for r6
classify_df = pl.read_csv('results_backup/classify/combined_classify.tsv.gz', separator='\t', ignore_errors=True)
print("Number of input sequences: ", classify_df.height)

Number of input sequences:  1261909


In [2]:
# number of sequences included in r6
seqhasher_df = pl.read_csv('results_backup/dereplicate/uhvdb_seqhasher.tsv.gz', separator='\t', ignore_errors=True)
print("Total r6 sequences: ", seqhasher_df.height)
print("Unique r6 sequences: ", seqhasher_df.n_unique('hash'))

Total r6 sequences:  1541381
Unique r6 sequences:  1155490


In [3]:
# number of genomovars in r6
genomovar_df = pl.read_csv('results_backup/dereplicate/new_unique_viruses.info.tsv.gz', separator='\t', ignore_errors=True)

print("Unique in r6: ", genomovar_df.n_unique('uhvdb_id'))
print("Genomovars in r6: ", genomovar_df.n_unique('cluster_id'))

Unique in r6:  1155490
Genomovars in r6:  925604


In [4]:
# number of species / genera in r5 / r6
v5_meta = pl.read_csv('uhvdb_v5_metadata_sra.tsv', separator='\t', ignore_errors=True)
species_df = pl.read_csv('results_backup/anicluster/new_genomovar_reps.info.tsv.gz', separator='\t', ignore_errors=True)
aaicluster_df = pl.read_csv('results_backup/aaicluster/uhvdb_aaicluster.tsv.gz', separator='\t', ignore_errors=True)

print("Species in r5: ", v5_meta.n_unique('species_rep'))
print("Species in r6: ", species_df.filter(pl.col('uhvdb_id').is_in(set(genomovar_df['genomovars_rep']))).n_unique('species_rep'))
print("Genera in r5: ", v5_meta.n_unique('genus_cluster_id'))
print("Genera in r6: ", aaicluster_df.n_unique('genus_cluster_id'))


Species in r5:  206289
Species in r6:  291605


In [6]:
# number subgenera - families in r6
aaicluster_df = pl.read_csv('results_backup/aaicluster/uhvdb_aaicluster.tsv.gz', separator='\t', ignore_errors=True)
print("Species in r6: ", aaicluster_df.n_unique('uhvdb_id'))
print("Subgenera in r6: ", aaicluster_df.n_unique('subgenus_cluster_id'))
print("Genera in r6: ", aaicluster_df.n_unique('genus_cluster_id'))
print("Subfamilies in r6: ", aaicluster_df.n_unique('subfamily_cluster_id'))
print("Families in r6: ", aaicluster_df.n_unique('family_cluster_id'))

Species in r6:  291605
Subgenera in r6:  133987
Genera in r6:  53506
Subfamilies in r6:  16401
Families in r6:  1590


### Source metadata + assemble v6 metadata table

Build `uhvdb_v6_metadata_sra.tsv` from v5 metadata + v6 `results_backup` tables:
remap clustering for all genomes, append newly mined sequences, fill available QC/taxonomy/CRISPR/PHIST fields, add source/SRA metadata for new DBs, and tag `added_in_release` (`r1`–`r6`). Lifestyle and protein-annotation columns stay blank for novel sequences.

Pipeline (implemented in `build_uhvdb_v6_metadata_sra.py`):
1. Remap all v5 rows onto v6 genomovar/species/AAI cluster IDs
2. Append new mined sequences from `results_backup/dereplicate/uhvdb_id_map.tsv.gz`
3. Fill classify / CheckV / taxonomy / CRISPR / PHIST where available; leave protein/lifestyle blank for novel genomes
4. Attach source + SRA metadata for VIRE / OAVGC / MAGIC / METAVR / NCBIVIRUS (MAGIC: link `assembly_group` → `Sample_ID` / `Sample_ID2` in `magic_sample_metadata.xlsx`, then resolve run/biosample/bioproject via the SRA parquet; fall back to bioproject from sequence meta)
5. Assign `body_site` from source metadata where possible: VIRE `microntology`, metaVR GOLD ecosystem fields (also used for IMGVR via shared UVIG IDs), NCBI `Isolation_Source` (keyword map to Gut / Airways / Skin / Urogenital / Blood / Other; OAVGC→Airways, MAGIC→Gut)
6. Tag `added_in_release` (`r1`–`r6`) by when each `seq_name` row entered the table (r1–r4 from historical seqhashers; leftover v5 rows → `r5`; new mined rows → `r6`)
7. Write `uhvdb_v6_metadata_sra.tsv`


In [ ]:
# Run full assembly (writes uhvdb_v6_metadata_sra.tsv). Re-run after protein ann. finish to fill remaining deferred columns.
from build_uhvdb_v6_metadata_sra import main as build_uhvdb_v6_metadata

build_uhvdb_v6_metadata()


In [ ]:
import polars as pl

uhvdb_v6_metadata = pl.read_csv(
    'uhvdb_v6_metadata_sra.tsv',
    separator='\t',
    infer_schema_length=5000,
    ignore_errors=True,
)
print('Total rows:', f"{uhvdb_v6_metadata.height:,}")
print(uhvdb_v6_metadata.group_by('added_in_release').len().sort('added_in_release'))
print(
    uhvdb_v6_metadata
    .filter(pl.col('added_in_release') == 'r6')
    .group_by('source_db')
    .agg(
        pl.len().alias('n'),
        pl.col('acc').is_not_null().sum().alias('n_acc'),
        pl.col('biosample').is_not_null().sum().alias('n_biosample'),
        pl.col('bioproject').is_not_null().sum().alias('n_bioproject'),
    )
    .sort('n', descending=True)
)


Total rows: 1,541,381
shape: (6, 2)
┌──────────────────┬────────┐
│ added_in_release ┆ len    │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ r1               ┆ 201324 │
│ r2               ┆ 2764   │
│ r3               ┆ 358705 │
│ r4               ┆ 197784 │
│ r5               ┆ 55741  │
│ r6               ┆ 725063 │
└──────────────────┴────────┘
shape: (5, 5)
┌───────────┬────────┬────────┬─────────────┬──────────────┐
│ source_db ┆ n      ┆ n_acc  ┆ n_biosample ┆ n_bioproject │
│ ---       ┆ ---    ┆ ---    ┆ ---         ┆ ---          │
│ str       ┆ u32    ┆ u32    ┆ u32         ┆ u32          │
╞═══════════╪════════╪════════╪═════════════╪══════════════╡
│ VIRE      ┆ 467579 ┆ 436128 ┆ 466556      ┆ 436128       │
│ OAVGC     ┆ 117211 ┆ 96565  ┆ 83746       ┆ 83746        │
│ NCBIVIRUS ┆ 73654  ┆ 6968   ┆ 12865       ┆ 6924         │
│ MAGIC     ┆ 46728  ┆ 18821  ┆ 17876       ┆ 37748        │
│ METAVR    ┆ 19891  ┆ 14704  ┆ 19203    

In [ ]:
# number of unique (by hash) NCBIVIRUS genomes in UHVDB r6
ncbi = uhvdb_v6_metadata.filter(pl.col('source_db') == 'NCBIVIRUS')
print(f'Unique NCBIVIRUS genomes (by hash) in UHVDB r6: {ncbi.n_unique("hash"):,}')


Unique NCBIVIRUS genomes (by hash) in UHVDB r6: 69,971
